# Robustez y prueba fuera de distribución

1. **Perturbaciones sintéticas** sobre el conjunto de prueba (brillo, desenfoque, ruido, JPEG, foto de pantalla).
2. **Fuera de distribución (B6):** fotos tomadas por el equipo (`data/ood/`) comparadas contra la prueba original.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.config import *
sns.set_theme(style="whitegrid")
FIG_DIR.mkdir(parents=True, exist_ok=True)
ARCH = DEFAULT_ARCH
N_SAMPLE = 450  # muestra fija de test (50 por clase) para que el notebook corra en minutos

In [2]:
import torch
from PIL import Image
from sklearn.metrics import f1_score
from src.evaluate import load_model
from src.data import get_transforms, CLASS_TO_IDX
from src.robustness import PERTURBATIONS, apply
model = load_model(ARCH, "cpu"); tf = get_transforms(train=False)
test = pd.read_csv(SPLITS_DIR / "test.csv")
sub = test.groupby("label").sample(N_SAMPLE // len(CLASSES), random_state=SEED)
imgs = [Image.open(ROOT / p).convert("RGB") for p in sub.path]
y = sub.label.map(CLASS_TO_IDX).to_numpy()

@torch.no_grad()
def f1_on(images):
    preds = []
    for i in range(0, len(images), 64):
        x = torch.stack([tf(im) for im in images[i:i + 64]])
        preds.append(model(x).argmax(1))
    return f1_score(y, torch.cat(preds).numpy(), average="macro")

In [3]:
rows = []
for kind, levels in PERTURBATIONS.items():
    for lv in levels:
        rows.append({"perturbación": kind, "nivel": lv, "f1_macro": f1_on([apply(im, kind, lv) for im in imgs])})
rob = pd.DataFrame(rows)
rob.to_csv(RUNS_DIR / ARCH / "robustez.csv", index=False)
rob.pivot(index="nivel", columns="perturbación", values="f1_macro").round(3)

perturbación,brillo,desenfoque,jpeg,pantalla,ruido
nivel,,,,,
0.0,NaN,0.882,NaN,0.882,0.882
0.4,0.864,NaN,NaN,NaN,NaN
0.6,0.879,NaN,NaN,NaN,NaN
1.0,0.882,0.855,NaN,0.875,NaN
1.4,0.867,NaN,NaN,NaN,NaN
1.8,0.821,NaN,NaN,NaN,NaN
2.0,NaN,0.816,NaN,0.776,NaN
3.0,NaN,0.741,NaN,0.640,NaN
5.0,NaN,0.470,0.514,NaN,NaN


In [4]:
fig, axes = plt.subplots(1, len(PERTURBATIONS), figsize=(18, 3.4))
for ax, (kind, g) in zip(axes, rob.groupby("perturbación", sort=False)):
    ax.plot(range(len(g)), g.f1_macro, "o-"); ax.set_xticks(range(len(g))); ax.set_xticklabels(g.nivel)
    ax.set_title(kind); ax.set_ylim(0, 1); ax.set_xlabel("nivel")
axes[0].set_ylabel("F1 macro")
plt.tight_layout(); plt.savefig(FIG_DIR / "robustez_perturbaciones.png", dpi=120); plt.show()

In [5]:
ex = imgs[0]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (k, lv) in zip(axes, [("brillo", 0.4), ("desenfoque", 3), ("ruido", 35), ("jpeg", 5), ("pantalla", 3)]):
    ax.imshow(apply(ex, k, lv)); ax.set_title(f"{k} = {lv}"); ax.axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "ejemplos_perturbaciones.png", dpi=100); plt.show()

## Fuera de distribución: otro conjunto de datos (ISIA Food-500)

80 imágenes de la partición de prueba de **ISIA Food-500** (Min et al., 2020), recolectadas de buscadores web (no de foodspotting.com como Food-101). Descarga y verificación de duplicados con Food-101 (pHash): `python -m src.download_ood`. Food-500 no incluye guacamole ni churros, así que la comparación justa es contra la prueba original **restringida a las mismas 7 clases**.

In [6]:
from sklearn.metrics import f1_score, accuracy_score
ood_pred = RUNS_DIR / ARCH / "eval_ood" / "predictions.csv"
if not ood_pred.exists():
    print("Aún no hay evaluación OOD: python -m src.download_ood && python -m src.evaluate --split ood")
else:
    od = pd.read_csv(ood_pred); te = pd.read_csv(RUNS_DIR / ARCH / "eval_test" / "predictions.csv")
    present = sorted(od.label.unique())
    te7 = te[te.label.isin(present)]
    def summ(d):
        return {"n": len(d), "exactitud": accuracy_score(d.label, d.pred),
                "f1_macro": f1_score(d.label, d.pred, average="macro", labels=present),
                "confianza_media": d.conf.mean()}
    comp = pd.DataFrame({"Prueba Food-101 (9 clases)": summ(te) | {"f1_macro": f1_score(te.label, te.pred, average="macro")},
                         "Prueba Food-101 (mismas 7 clases)": summ(te7),
                         "Food-500 (OOD)": summ(od)}).T.round(3)
    comp.to_csv(RUNS_DIR / ARCH / "comparacion_ood.csv")
    display(comp)
    rec = pd.DataFrame({"recall Food-101": te7.assign(a=te7.label == te7.pred).groupby("label").a.mean(),
                        "recall Food-500": od.assign(a=od.label == od.pred).groupby("label").a.mean()}).round(2)
    rec.index = rec.index.map(DISPLAY); display(rec)
    ax = rec.plot.bar(figsize=(9, 3.8), ylim=(0, 1.05)); ax.set_title("Recall por clase: prueba original vs. otro dataset")
    plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.savefig(FIG_DIR / "ood_vs_test.png", dpi=120); plt.show()
    print("Errores en OOD:"); display(od[od.label != od.pred].assign(real=lambda d: d.label.map(DISPLAY), predicho=lambda d: d.pred.map(DISPLAY))[["real", "predicho", "conf", "food500_cat"]])

,n,exactitud,f1_macro,confianza_media
Prueba Food-101 (9 clases),2374.0,0.856,0.859,0.768
Prueba Food-101 (mismas 7 clases),1874.0,0.828,0.837,0.748
Food-500 (OOD),80.0,0.750,0.767,0.724


,recall Food-101,recall Food-500
label,,
Burrito,0.82,0.7
Ceviche,0.88,0.9
Quesadilla,0.88,0.6
Huevos rancheros,0.83,0.7
Nachos,0.85,0.9
Otro platillo,0.73,0.6
Tacos,0.87,1.0


Errores en OOD:


,real,predicho,conf,food500_cat
19,Nachos,Otro platillo,0.914880,Nachos
20,Huevos rancheros,Tacos,0.511427,Huevos_rancheros
22,Huevos rancheros,Ceviche,0.370625,Huevos_rancheros
27,Huevos rancheros,Tacos,0.888885,Huevos_rancheros
30,Quesadilla,Huevos rancheros,0.832993,Quesadilla
33,Quesadilla,Ceviche,0.423839,Quesadilla
35,Quesadilla,Tacos,0.428488,Quesadilla
36,Quesadilla,Guacamole,0.399601,Quesadilla
46,Burrito,Tacos,0.540499,Burrito
47,Burrito,Huevos rancheros,0.407099,Burrito
